# 00 · Cuaderno maestro / *Master notebook*
**GUAIRA-SAR** · SRL *2026 Venezuela, Colombia and Panama Earthquakes*

🇪🇸 **Un solo cuaderno** que llama a todo el flujo (inventario → descargas → InSAR → daño → figuras) usando el código de GitHub (`src/`). Cada paso se ejecuta **una vez**: queda registrado en Drive (`05_documentacion/pasos_ejecutados.csv`) y se salta solo la próxima vez. Los pasos ya hechos el 19-nov figuran como ✅ *previo*. Para repetir uno a propósito: `P("id", forzar=True)`.
🇬🇧 **One notebook** that calls the whole workflow using the GitHub code. Each step runs **once**; it is logged on Drive and skipped next time. Steps done on 19 Nov show as ✅ *previo*. To repeat on purpose: `P("id", forzar=True)`.

⚠️ 🇪🇸 **HyP3:** el envío identifica trabajos por *par de escenas + resolución* (no por nombre) y **solo envía lo que no existe**. Hoy solo falta 033A a 40 m.
⚠️ 🇬🇧 **HyP3:** submissions are matched by *scene pair + resolution*; only what does not exist is sent. Today only 033A at 40 m is missing.
🔒 🇪🇸 Credenciales en **Secretos de Colab** (🔑 en la barra lateral): `EARTHDATA_USER` y `EARTHDATA_PASS`. No se escriben en el cuaderno. 🇬🇧 Credentials in **Colab Secrets**; never typed into the notebook.
📝 Contexto completo / full context: `docs/HANDOFF.md`.

## 0 · Preparar entorno / *Setup*

In [1]:
NOTEBOOK = "00_Maestro"
!pip install -q asf_search hyp3_sdk rasterio geopandas shapely contextily pyarrow duckdb boto3 pystac-client requests h5py pyproj scipy huggingface_hub cmcrameri
import os, subprocess, datetime
REPO = "/content/SAR_multifrequency"
CUADERNO_VERSION = "2026-11-27.5"      # versión de ESTE cuaderno / version of THIS notebook
def _git(*a):
    return subprocess.run(["git", "-C", REPO, *a], capture_output=True, text=True).stdout.strip()
if os.path.isdir(REPO):
    _antes = _git("rev-parse", "HEAD")
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
    _log = _git("log", "--oneline", "--no-merges", f"{_antes}..HEAD") if _antes else ""
    print("Cambios nuevos en GitHub desde la última vez / new GitHub changes since last time:")
    print(_log[:1800] if _log else "   (ninguno / none)")
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/jf-floresriera/SAR_multifrequency", REPO], check=True)
_vf = f"{REPO}/notebooks/VERSION_CUADERNO.txt"
_v = open(_vf).read().strip() if os.path.exists(_vf) else ""
if _v and _v != CUADERNO_VERSION:
    print()
    print(f"⚠️ ESTE cuaderno es la versión {CUADERNO_VERSION} y en GitHub hay la {_v}: puede faltar alguna celda nueva.")
    print("   Ábralo de nuevo desde / reopen from: https://colab.research.google.com/github/jf-floresriera/SAR_multifrequency/blob/main/notebooks/00_Maestro.ipynb")
    print("   Mientras tanto, cualquier paso se corre en una celda nueva con: P(\"id_del_paso\", forzar=True)  (ids al final / ids at the end)")
SRC = f"{REPO}/src"
VERSION = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()

from google.colab import drive
drive.mount("/content/drive")
RAIZ = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR"
TRABAJO = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/inventario_2026"
for d in (TRABAJO, f"{RAIZ}/05_documentacion"):
    os.makedirs(d, exist_ok=True)
os.chdir(TRABAJO)
with open(f"{RAIZ}/05_documentacion/versiones_codigo.csv", "a") as f:   # trazabilidad / traceability
    f.write(f"{datetime.datetime.now():%Y-%m-%d %H:%M},{NOTEBOOK},{VERSION}\n")
import sys, importlib
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import comun, maestro; importlib.reload(comun); importlib.reload(maestro)
LOGS = f"{RAIZ}/05_documentacion/logs"
def correr(script):
    """Ejecuta src/<script> y guarda todo lo impreso en 05_documentacion/logs / runs and logs everything."""
    return comun.correr(script, LOGS, SRC, NOTEBOOK, VERSION)
def P(pid, **kw):
    """Ejecuta un paso del flujo salvo que ya esté hecho / run a workflow step unless already done."""
    return maestro.paso(pid, correr, RAIZ, VERSION, **kw)
print(f"Código / code: {VERSION} · Carpeta / folder: {os.getcwd()}")
print("Pasos disponibles / available steps:", maestro.IDS)

Mounted at /content/drive
Código / code: 1338141 · Carpeta / folder: /content/drive/My Drive/InSAR_LaGuaira_Resultados/inventario_2026
Pasos disponibles / available steps: ['inv_umbra', 'inv_nisar', 'inv_s1', 'inv_alos2', 'inv_tabla', 'mapas_base', 'hyp3_estado', 'hyp3_envio', 'descargas', 'auxiliares', 'nisar_tif', 's1_tif', 'descomp', 'ucsd_los', 'auditoria', 'qc', 'mosaico', 'comparacion', 'insar_metricas', 'inventario', 'tablas', 'danio_ref', 'vantor', 'danio_sitios', 'exportar']


## 1 · Credenciales NASA Earthdata (Secretos de Colab) / *Credentials (Colab Secrets)*

In [2]:
import asf_search as asf
try:
    from google.colab import userdata
    usuario, clave = userdata.get("EARTHDATA_USER"), userdata.get("EARTHDATA_PASS")
except Exception:
    import getpass
    print("Sin Secretos de Colab: se pedirán aquí (la clave no se muestra). / No Colab Secrets: prompting here.")
    usuario = input("Usuario Earthdata / username: "); clave = getpass.getpass("Contraseña / password: ")
session = asf.ASFSession().auth_with_creds(usuario, clave)
print("Autenticación correcta / authenticated")

Autenticación correcta / authenticated


## 2 · Panel de estado / *Status board*
✅ hecho · ♻️ se puede correr siempre · ⏳ pendiente

In [3]:
import pandas as pd; pd.set_option("display.max_colwidth", 90, "display.width", 200)
maestro.estado(RAIZ)

,paso / step,fase / phase,estado / status,descripción (ES),description (EN),detalle / detail
0,inv_umbra,A Inventario,✅,"Umbra X: 26 adquisiciones, STAC, huellas","Umbra X: 26 collects, STAC, footprints",previo / earlier · 2026-11-19 · carpeta/folder inventario_2026/umbra_inventario
1,inv_nisar,A Inventario,✅,NISAR L: GUNW cosísmicos,NISAR L: coseismic GUNW,previo / earlier · 2026-11-19 · inventario_2026/nisar_filtrado/nisar_productos.csv
2,inv_s1,A Inventario,✅,Sentinel-1 C: pares cosísmicos (solo búsqueda),Sentinel-1 C: coseismic pairs (search only),"previo / earlier · 2026-11-19 · trabajos/jobs HyP3 025D, Path33"
3,inv_alos2,A Inventario,✅,ALOS-2 L: búsqueda en ASF,ALOS-2 L: ASF search,2026-11-26 19:40 · alos2_buscar.py
4,inv_tabla,A Inventario,✅,Tabla maestra y línea de tiempo (suplemento),Master table and timeline (supplement),previo / earlier · 2026-11-19 · cuaderno/notebook Inventario_Multisensor (timeline)
5,mapas_base,A Inventario,✅,Mapas base y figura de ubicación (Fig. 1),Base maps and location figure (Fig. 1),2026-11-26 22:13 · mapas_base.py md5:d9091a94
6,hyp3_estado,B Descargas,✅,Estado de HyP3 y descarga de lo terminado,HyP3 status and download of finished jobs,2026-11-26 22:13 · hyp3_estado.py md5:6847f2f9
7,hyp3_envio,B Descargas,⏳,Enviar a HyP3 (SOLO lo que no existe; gasta créditos),Submit to HyP3 (ONLY what does not exist; uses credits),pendiente / pending
8,descargas,B Descargas,✅,"NISAR, Umbra GEC y Sentinel-1 a Drive","NISAR, Umbra GEC and Sentinel-1 to Drive",2026-11-26 22:18 · descargas.py md5:9eb5c1c2
9,auxiliares,C InSAR,✅,Réplicas USGS y WorldCover,USGS aftershocks and WorldCover,2026-11-26 20:33 · auxiliares.py


## A · Inventario multisensor / *Multi-sensor inventory*
🇪🇸 Umbra (X), NISAR y ALOS-2 (L), Sentinel-1 (C), Planet. Estas figuras y tablas van al **material suplementario** (S1–S3); una versión sintética alimenta la Fig. 1. 🇬🇧 These go to the **electronic supplement**.

In [4]:
P("inv_umbra")
P("inv_nisar")

⏭️  inv_umbra: ya hecho / already done. Use forzar=True para repetir / to repeat.
⏭️  inv_nisar: ya hecho / already done. Use forzar=True para repetir / to repeat.


In [5]:
P("inv_s1", variables={"ENVIAR_HYP3": False})   # solo busca / search only

⏭️  inv_s1: ya hecho / already done. Use forzar=True para repetir / to repeat.


In [6]:
P("inv_alos2")   # ALOS-2: si ASF no lo tiene, ver docs/HANDOFF.md §5 (fecha de corte 5-dic) / see HANDOFF §5

⏭️  inv_alos2: ya hecho / already done. Use forzar=True para repetir / to repeat.


In [7]:
P("inv_tabla")
P("mapas_base")   # se repite solo si cambia su código / re-runs by itself when its code changes

⏭️  inv_tabla: ya hecho / already done. Use forzar=True para repetir / to repeat.
⏭️  mapas_base: ya hecho / already done. Use forzar=True para repetir / to repeat.


## B · HyP3 y descargas / *HyP3 and downloads*
🇪🇸 `hyp3_estado` descarga lo terminado a Drive (HyP3 borra a los ~14 días). `hyp3_envio` está **bloqueado** hasta poner `confirmar=True`: enviará SOLO lo que no exista (hoy: 033A a 40 m, 1 trabajo). 🇬🇧 `hyp3_envio` is locked until `confirmar=True`.

In [8]:
P("hyp3_estado", variables={"DESCARGAR": True})

                  nombre    estado             enviado     expira
       S1_033A_cosismico SUCCEEDED 2026-11-19 22:07:19 2026-12-04
       S1_025D_cosismico SUCCEEDED 2026-11-19 19:23:52 2026-12-04
       S1_025D_cosismico SUCCEEDED 2026-11-19 19:21:33 2026-12-04
       S1_025D_cosismico SUCCEEDED 2026-11-19 19:13:32 2026-12-04
       S1_025D_cosismico SUCCEEDED 2026-11-19 18:57:19 2026-12-04
Sismo_LaGuaira_S1_Path33 SUCCEEDED 2026-11-05 17:42:57 2026-11-20
Sismo_LaGuaira_S1_Path33 SUCCEEDED 2026-11-05 17:39:53 2026-11-20

SUCCEEDED = listo/ready · RUNNING = procesando/processing · PENDING = en cola/queued · FAILED = falló/failed
   ya descargado / already downloaded: Sismo_LaGuaira_S1_Path33 (S1AD_20260618T224250_20260625T224209_VVP007_INT80_G_ueF_262A.zip)
   ya descargado / already downloaded: S1_025D_cosismico (S1AD_20260618T101640_20260625T101600_VVP007_INT40_G_ueF_2AEE.zip)
   ya descargado / already downloaded: S1_033A_cosismico (S1AD_20260618T224250_20260625T224209_VVP007_INT40

True

In [9]:
P("hyp3_envio", confirmar=False, variables={"ENVIAR_HYP3": True})   # ← cambiar a confirmar=True para enviar / set True to submit

🛑 hyp3_envio gasta créditos de HyP3. Revise primero con inv_s1 y repita con confirmar=True.
🛑 hyp3_envio uses HyP3 credits. Check with inv_s1 first, then repeat with confirmar=True.
   Solo se enviará lo que NO exista (par de escenas + resolución). / Only what does not exist (scene pair + resolution) is submitted.


In [10]:
P("descargas", variables={"ESPERAR_HYP3": False})   # NISAR (ya en Drive se omite), Umbra GEC, Sentinel-1

NISAR: 6 cosísmicos / coseismic, faltan / missing 0
Umbra: 26 GEC, 21.63 GB


  0%|          | 0/4 [00:00<?, ?it/s]

S1AD_20260618T101640_20260625T101600_VVP007_INT40_G_ueF_2AEE.zip:   0%|          | 0/1109735086 [00:00<?, ?it/…

S1AD_20260618T101640_20260625T101600_VVP007_INT40_G_ueF_EDD4.zip:   0%|          | 0/1109728063 [00:00<?, ?it/…

S1AD_20260618T101640_20260625T101600_VVP007_INT40_G_ueF_E245.zip:   0%|          | 0/1109723987 [00:00<?, ?it/…

S1AD_20260618T101640_20260625T101600_VVP007_INT40_G_ueF_0B32.zip:   0%|          | 0/1109734162 [00:00<?, ?it/…

  0%|          | 0/1 [00:00<?, ?it/s]

S1AD_20260618T224250_20260625T224209_VVP007_INT40_G_ueF_9346.zip:   0%|          | 0/1223884675 [00:00<?, ?it/…

Listo / Done -> 05_documentacion/registro_descargas.csv

📝 Registro / log: /content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/05_documentacion/logs/20261127_125302_00_Maestro_descargas.log


True

## C · InSAR banda C y L / *C- and L-band InSAR*
🇪🇸 Escriba las coordenadas de CCS1 (ficha NGL / IGVSB) para validar con GNSS. 🇬🇧 Enter CCS1 coordinates to validate.

In [11]:
P("auxiliares")
P("nisar_tif")
P("s1_tif")

⏭️  auxiliares: ya hecho / already done. Use forzar=True para repetir / to repeat.
⏭️  nisar_tif: ya hecho / already done. Use forzar=True para repetir / to repeat.
⏭️  s1_tif: ya hecho / already done. Use forzar=True para repetir / to repeat.


### C-pre · Pares PRE-sísmicos (control para el daño por coherencia) / Pre-seismic control pairs
🇪🇸 1) Primera corrida **sin** confirmar: muestra el par Sentinel-1 propuesto por track, los créditos y el costo (15 por par); baja y convierte los GUNW PRE-sísmicos de NISAR (gratis). 2) Si está de acuerdo, repita con `confirmar=True` para enviar el par a HyP3. 3) ~40 min después: `P("hyp3_estado")` y `P("s1_tif", forzar=True)` (el par pre va solo a `02_procesados/sentinel1_pre`). Nunca reenvía un par que ya exista.
🇬🇧 1) First run WITHOUT confirming: shows the proposed S-1 pair per track, credits and cost; downloads/converts NISAR pre-seismic GUNW (free). 2) Re-run with `confirmar=True` to submit. 3) ~40 min later: `hyp3_estado` and `s1_tif`. Never resubmits an existing pair.

In [ ]:
P("presismico")                    # 1) revisar / review
# P("presismico", confirmar=True)  # 2) enviar a HyP3 / submit (15 créditos / credits)

In [12]:
P("descomp", variables={"CCS1_LON": None, "CCS1_LAT": None})   # ej./e.g. -66.88, 10.49 (verificar)

🔄 descomp: el código cambió desde la última vez: se repite solo / code changed since the last run: re-running
NISAR: ['NISAR_T054D_F083_20260617_20260629', 'NISAR_T061A_F007_20260618_20260630', 'NISAR_T126D_F083_20260622_20260704', 'NISAR_T126D_F084_20260622_20260704', 'NISAR_T162A_F006_20260613_20260625', 'NISAR_T162A_F007_20260613_20260625']
   ionosfera CORREGIDA / ionosphere CORRECTED: NISAR_T054D_F083_20260617_20260629
   referencia / reference: NISAR_T054D_F083_20260617_20260629  desfase restado / offset removed = +249 mm
   ionosfera CORREGIDA / ionosphere CORRECTED: NISAR_T061A_F007_20260618_20260630
   referencia / reference: NISAR_T061A_F007_20260618_20260630  desfase restado / offset removed = -616 mm
   ionosfera CORREGIDA / ionosphere CORRECTED: NISAR_T126D_F083_20260622_20260704
   referencia / reference: NISAR_T126D_F083_20260622_20260704  desfase restado / offset removed = -517 mm
   ionosfera CORREGIDA / ionosphere CORRECTED: NISAR_T126D_F084_20260622_20260704
   [!] S

True

### C0 · Auditoría de correcciones / Corrections audit
🇪🇸 LEE cada GUNW de NISAR y cada ZIP de HyP3 y lista qué correcciones trae (ionosfera, troposfera, mareas, DEM…). Ver docs/CORRECCIONES_INSAR.md.
🇬🇧 READS every NISAR GUNW and HyP3 ZIP and lists which corrections it carries. See docs/CORRECCIONES_INSAR.md.

In [13]:
P("auditoria")

         sensor                                  capa_o_correccion                            presente                                                                                    ejemplo
     NISAR GUNW        BANDERA azimuthIonosphericCorrectionApplied                                True                                                                                           
     NISAR GUNW             BANDERA commonBandAzimuthFilterApplied                               False                                                                                           
     NISAR GUNW               BANDERA commonBandRangeFilterApplied                               False                                                                                           
     NISAR GUNW               BANDERA ellipsoidalFlatteningApplied                                True                                                                                           
     NISAR GUNW   BANDERA hydr

True

In [ ]:
P("verif_iono")   # Punto B: ¿la ionosfera ya viene aplicada en NISAR? / is the ionosphere already applied?

### 🔁 Reproducir TODO (paneles individuales **y** mosaicos) / Reproduce EVERYTHING (individual panels **and** mosaics)
🇪🇸 Orden: `nisar_tif`/`s1_tif` → `descomp` → `auditoria` → `ucsd_los` → **`mosaico`** → `qc` → `comparacion` → `insar_metricas`.
- **Paneles individuales (se conservan):** `qc` (una figura por interferograma: LOS en mm, fase en rad, franjas, coherencia, histogramas) e `insar_metricas` (`insar_comparable_propios_*`, `insar_comparable_ucsd_*`, perfil, tablas de métricas).
- **Mosaicos (adicionales):** `mosaico` (une frames/tracks vecinos de la misma familia y dirección; escribe también la procedencia de cada pieza), el `qc` de cada mosaico (`qc_*_MOSAICO_*`) y `comparacion` (figura principal con todos los modelos).
- Texto principal = mosaicos + comparación; **suplemento = paneles individuales y QC**.
🇬🇧 Individual panels (kept): `qc` and `insar_metricas`. Mosaics (additional): `mosaico`, mosaic QC and `comparacion`. Main text = mosaics + comparison; supplement = individual panels and QC.

### C1b · ALOS-2 y NISAR procesados por UCSD / UCSD-processed ALOS-2 and NISAR
🇪🇸 Descarga 7 archivos de topex.ucsd.edu (~445 MB cada uno, ~3 GB en total; se guardan en 01_datos_crudos/ucsd_los) y los convierte a GeoTIFF sobre nuestra malla. Decide el signo comparando con NUESTRO NISAR. **No redistribuir** (licencia no declarada): citar a UCSD/Scripps. Tarda unos minutos.
🇬🇧 Downloads 7 files from topex.ucsd.edu (~445 MB each, ~3 GB total) and converts them to GeoTIFF on our grid; sign decided by comparison with OUR NISAR. **Do not redistribute** (licence not stated): cite UCSD/Scripps.

In [14]:
P("ucsd_los")   # prueba rápida / quick test: variables={"SOLO_TRACKS": ["NSR_T162A"]}

⏭️  ucsd_los: ya hecho / already done. Use forzar=True para repetir / to repeat.


### C1c · Mosaico de frames/tracks NISAR vecinos / Mosaic of adjacent NISAR frames/tracks
🇪🇸 Une los frames y tracks de la misma dirección (ASC: T162A+T061A; DESC: T054D+T126D, como hace UCSD), **igualando el desfase en el solape**, y mide la costura (RMS). Es para VER y comparar; Este/Vertical sigue usando los productos separados.
🇬🇧 Merges same-direction frames/tracks (ASC: T162A+T061A; DESC: T054D+T126D, as UCSD does), **matching the offset in the overlap**, and reports the seam (RMS). For viewing/comparison; East/Up still uses the separate products.

In [15]:
P("mosaico")


== ALOS-2 (UCSD) ASC: ['UCSD_ALOS2_T040A_20250114_20260630']
   (un solo producto: no hay vecino que unir / single product: nothing to merge)

== ALOS-2 (UCSD) DESC: ['UCSD_ALOS2_T134D_20260610_20260708', 'UCSD_ALOS2_T135D_20260615_20260629']
   UCSD_ALOS2_T135D_20260615_20260629: solape 1,215,710 px · desfase -36 mm (solape) · RMS solape 34 mm

== NISAR ASC: ['NISAR_T162A_F007_20260613_20260625', 'NISAR_T061A_F007_20260618_20260630', 'NISAR_T162A_F006_20260613_20260625']
   NISAR_T061A_F007_20260618_20260630: solape 116,541 px · desfase -516 mm (solape) · RMS solape 147 mm
   NISAR_T162A_F006_20260613_20260625: solape 61,361 px · desfase +125 mm (solape) · RMS solape 171 mm

== NISAR DESC: ['NISAR_T126D_F083_20260622_20260704', 'NISAR_T054D_F083_20260617_20260629', 'NISAR_T126D_F084_20260622_20260704']
   NISAR_T054D_F083_20260617_20260629: solape 103,160 px · desfase +157 mm (solape) · RMS solape 44 mm
   NISAR_T126D_F084_20260622_20260704: solape 76,536 px · desfase -620 mm (solape

True

### C0b · Control de calidad de cada interferograma / Per-interferogram QC
🇪🇸 ¿La deformación realmente se ve? Muestra LOS con estiramiento robusto, **franjas** (fase re-envuelta), coherencia, histograma, y escribe un diagnóstico por producto. Ejecutar ANTES de interpretar mapas.
🇬🇧 Is the deformation really visible? Robust-stretch LOS, **fringes** (re-wrapped phase), coherence, histogram and a per-product diagnosis. Run BEFORE interpreting maps.

In [16]:
P("qc")

Máscara de tierra (sin mar ni lagos): 56 % de la ventana / land without sea and lakes
   barrido de coherencia / sweep: {'cobertura_coh0.1_pct': 51.1, 'ruido_coh0.1_mm': 16.6, 'cobertura_coh0.2_pct': 39.8, 'ruido_coh0.2_mm': 17.2, 'cobertura_coh0.3_pct': 21.6, 'ruido_coh0.3_mm': 18.5, 'cobertura_coh0.4_pct': 13.0, 'ruido_coh0.4_mm': 22.9}
S1_20260618_20260625_0B32: p2/p98 = -111/+170 mm · franjas ≈ 10.1 · SNR ≈ 7.6 · 44 % de la huella es MAR (incoherente): la cobertura global subestima la de tierra (39 % de la tierra supera coh ≥ 0.3); solo 27 % de los píxeles supera coh ≥ 0.3
   barrido de coherencia / sweep: {'cobertura_coh0.1_pct': 27.4, 'ruido_coh0.1_mm': 8.4, 'cobertura_coh0.2_pct': 21.2, 'ruido_coh0.2_mm': 8.3, 'cobertura_coh0.3_pct': 11.5, 'cobertura_coh0.4_pct': 7.1}
S1_20260618_20260625_9346: p2/p98 = -89/+265 mm · franjas ≈ 12.8 · SNR ≈ 27.9 · 44 % de la huella es MAR (incoherente): la cobertura global subestima la de tierra (21 % de la tierra supera coh ≥ 0.3); solo 17 % de 

True

### C1d · Figura de comparación de todos los modelos / Comparison figure of all models
🇪🇸 Filas = familia de sensor (Sentinel-1, NISAR, NISAR-UCSD, ALOS-2-UCSD), columnas = ASC | DESC; misma ventana y escala. **Figura principal**; los productos individuales y el QC van al suplemento.
🇬🇧 Rows = sensor family, columns = ASC | DESC; same window and scale. **Main figure**; individual products and QC go to the supplement.

In [17]:
P("comparacion")   # Fig. 3 (franjas) y Fig. 4 (LOS) · regla de selección: variables={"UMBRAL_COSTURA_MM": 40, "LIM_MM": 400}

=== SELECCIÓN DE PANELES / PANEL SELECTION ===
  familia direccion                  eleccion                           producto  cobertura_tierra_pct                                                                                                                       motivo
ALOS2UCSD       ASC            producto único UCSD_ALOS2_T040A_20250114_20260630                  37.4                                                                                 un solo producto en esta familia y dirección
ALOS2UCSD      DESC                   mosaico                            mosaico                  58.8                                                                    mosaico de 2 productos; MAD máx. de costura 30 mm ≤ 40 mm
    NISAR       ASC                   mosaico                            mosaico                  65.5                                                                    mosaico de 3 productos; MAD máx. de costura 33 mm ≤ 40 mm
    NISAR      DESC mejor producto indivi

True

### C2 · Métricas y gráficos comparables de toda la interferometría / Comparable metrics and plots for all InSAR
🇪🇸 Misma ventana, misma escala de color (± cm común) y mismo contexto para todos los productos (Sentinel-1 y NISAR, incluido el par pre-sísmico de control). Genera la tabla de métricas (CSV ES/EN), mapas regional y costa, y un perfil común.
🇬🇧 Same window, same colour scale (common ± cm) and same context for every product (Sentinel-1 and NISAR, including the pre-seismic control pair). Produces the metrics table (CSV ES/EN), regional and coastal maps, and a common profile.

In [18]:
P("insar_metricas")   # opcional / optional: variables={"LIM_MM": 300, "COH_MIN": 0.3}

    fuente                 sensor track        ref        sec relacion
0   propio             Sentinel-1       2026-06-18 2026-06-25       CO
1   propio             Sentinel-1       2026-06-18 2026-06-25       CO
2   propio                  NISAR  054D 2026-06-17 2026-06-29       CO
3   propio                  NISAR  061A 2026-06-18 2026-06-30       CO
4   propio                  NISAR  126D 2026-06-22 2026-07-04       CO
5   propio                  NISAR  126D 2026-06-22 2026-07-04       CO
6   propio                  NISAR  162A 2026-06-13 2026-06-25       CO
7   propio                  NISAR  162A 2026-06-13 2026-06-25       CO
8   propio  ALOS-2 (UCSD) mosaico   ASC 2025-01-14 2026-06-30       CO
9   propio   NISAR (UCSD) mosaico   ASC 2026-06-13 2026-06-30       CO
10  propio          NISAR mosaico   ASC 2026-06-13 2026-06-30       CO
11  propio     Sentinel-1 mosaico   ASC 2026-06-18 2026-06-25       CO
12  propio  ALOS-2 (UCSD) mosaico  DESC 2026-06-10 2026-07-08       CO
13  pr

/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encountered
  perf = np.nanmedian(a, axis=1)
/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encountered
  perf = np.nanmedian(a, axis=1)
/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encountered
  perf = np.nanmedian(a, axis=1)
/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encountered
  perf = np.nanmedian(a, axis=1)
/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encountered
  perf = np.nanmedian(a, axis=1)
/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encountered
  perf = np.nanmedian(a, axis=1)
/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encountered
  perf = np.nanmedian(a, axis=1)
/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encou

[('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/insar_perfil_comparable_es.pdf', 0.03), ('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/insar_perfil_comparable_es.tiff', 1.48), ('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/insar_perfil_comparable_es.png', 0.49)]
[('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/insar_comparable_propios_region_en.pdf', 5.71), ('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/insar_comparable_propios_region_en.tiff', 10.34), ('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/insar_comparable_propios_region_en.png', 2.35)]
[('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/insar_comparable_propios_costa_en.pdf', 2.11), ('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/ins

/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encountered
  perf = np.nanmedian(a, axis=1)
/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encountered
  perf = np.nanmedian(a, axis=1)
/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encountered
  perf = np.nanmedian(a, axis=1)
/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encountered
  perf = np.nanmedian(a, axis=1)
/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encountered
  perf = np.nanmedian(a, axis=1)
/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encountered
  perf = np.nanmedian(a, axis=1)
/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encountered
  perf = np.nanmedian(a, axis=1)
/content/SAR_multifrequency/src/insar_metricas.py:285: RuntimeWarning: All-NaN slice encou

[('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/insar_perfil_comparable_en.pdf', 0.03), ('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/insar_perfil_comparable_en.tiff', 1.47), ('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/insar_perfil_comparable_en.png', 0.48)]

Tabla / table: /content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/02_procesados/metricas_insar/metricas_insar_{es,en}.csv · figuras / figures: /content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/insar_*_{es,en}.*

📝 Registro / log: /content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/05_documentacion/logs/20261127_131621_00_Maestro_insar_metricas.log


True

### C3 · Tablas del artículo / Paper tables
🇪🇸 Tabla 1 (datos), Tabla 2 (procesamiento y correcciones), S1 (calidad con **tierra vs mar**), S2 (mosaicos), S3 (C vs L en píxeles comunes sobre tierra; propio vs UCSD). CSV + Markdown, ES y EN, en `05_documentacion/tablas/`.
🇬🇧 Tables 1, 2, S1 (quality, land vs sea), S2 (mosaics), S3 (C vs L on common land pixels; ours vs UCSD). CSV + Markdown, ES and EN.

In [19]:
P("tablas")

✓ Tabla1_datos: 23 filas / rows
✓ Tabla2_procesamiento: 23 filas / rows
✓ TablaS1_calidad: 16 filas / rows
✓ TablaS2_mosaicos: 15 filas / rows
✓ TablaS3_comparaciones: 6 filas / rows

Tablas / tables: /content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/05_documentacion/tablas

📝 Registro / log: /content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/05_documentacion/logs/20261127_132835_00_Maestro_tablas_articulo.log


True

### C4 · Figura de inventario (Fig. 2) / Inventory figure
🇪🇸 Línea de tiempo por banda (Umbra X, Sentinel-1 C, NISAR L, **ALOS-2 L**, óptico) con escenas y pares usados. Lista completa de figuras: `docs/LISTA_FIGURAS.md`.
🇬🇧 Timeline per band incl. ALOS-2. Full figure list: `docs/LISTA_FIGURAS.md`.

In [20]:
P("inventario")

[('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/fig_inventario_es.pdf', 0.02), ('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/fig_inventario_es.tiff', 0.41), ('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/fig_inventario_es.png', 0.08)]
[('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/fig_inventario_en.pdf', 0.02), ('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/fig_inventario_en.tiff', 0.4), ('/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/04_figuras/borradores/fig_inventario_en.png', 0.08)]

📝 Registro / log: /content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/05_documentacion/logs/20261127_132837_00_Maestro_fig_inventario.log


True

In [ ]:
P("cobertura")   # mapa de huellas y geometría (vuelo, mirada, incidencia) / footprints and geometry

## D · Daño: Umbra X + Planet + referencia / *Damage*
🇪🇸 Las figuras por sitio del 19-nov **no son publicables** (jet, título interno, Esri): tras actualizar `umbra_planet_sitios.py` al estilo nuevo use `forzar=True`. 🇬🇧 The 19 Nov per-site figures are not publishable; re-run with `forzar=True` after the style update.

### D2 · Óptico de alta resolución antes/después (Vantor abierto) / High-resolution optical before/after (open Vantor)
🇪🇸 Busca escenas de ~0,4 m anteriores y posteriores al sismo sobre cada sitio (sin descargas completas) y guarda un CSV y recortes de prueba. Licencia CC BY-NC 4.0: no subir a GitHub.
🇬🇧 Finds ~0.4 m pre- and post-event scenes over each site (no full downloads) and saves a CSV and test chips. CC BY-NC 4.0: do not commit.

In [21]:
P("vantor", variables={"SITIOS_VANTOR": {"Caracas_Este_Petare": (10.4835, -66.8070)},   # (lat, lon) ← cambiar por el centro real / replace with true centre
                                     "LADO_M": 1000})

⏭️  vantor: ya hecho / already done. Use forzar=True para repetir / to repeat.


In [22]:
P("danio_ref")

⏭️  danio_ref: ya hecho / already done. Use forzar=True para repetir / to repeat.


In [23]:
P("danio_sitios", variables={"LADO_M": 1000, "ZOOM_M": 400, "SITIOS_A_PROCESAR": ["Catia_La_Mar", "Playa_Grande_Urimare", "La_Guaira"]})

⏭️  danio_sitios: ya hecho / already done. Use forzar=True para repetir / to repeat.


## E · Figuras y tablas finales / *Final figures and tables*
🇪🇸 **Texto principal (≤ 10 figuras, ≤ 3 tablas):** Fig. 1 ubicación y cobertura · Fig. 3–5 LOS y Este/Vertical C vs L · Fig. 6 coherencia por cobertura · Fig. 7 ventana postsísmica · Fig. 8 daño Umbra × Planet · Fig. 9 daño vs gradiente/distancia a la falla · Fig. 10 infraestructura · Tablas 1–3.
**Suplemento (sin límite):** S1 línea de tiempo · S2 huellas por sensor (Sentinel-1, NISAR, ALOS-2, Umbra, Planet) · Tabla S1 inventario completo · S3 los 6 sitios de daño · S4 paletas y pruebas de gris.
🇬🇧 Scripts for Figs. 6, 7, 9, 10 and Table 3 are **pending**; they will be added to `src/` and as steps of this notebook (`PASOS` in `src/maestro.py`).

In [24]:
P("exportar")   # figuras a GitHub: NO subir imágenes Planet (CC BY-NC) ni Esri / do not upload Planet or Esri imagery

✓ prelim_s1_coherencia_fase_los.jpg (1800, 540)
✓ prelim_s1_zooms_guaira.jpg (1800, 1025)
✓ prelim_s1_correlacion_dem.jpg (1800, 1080)
✓ prelim_s1_viaducto.jpg (1800, 893)
✓ cobertura_nisar_umbra_es.jpg (1800, 1772)
✓ mapa_umbra_es.jpg (1800, 1145)
✓ linea_tiempo_es.jpg (1800, 782)
✓ sitio_catia_la_mar_es.jpg (1800, 822)
✓ sitio_playa_grande_es.jpg (1800, 822)
✓ sitio_la_guaira_es.jpg (1800, 1048)
✓ sitio_caraballeda_es.jpg (1463, 1681)
✓ cobertura_nisar_umbra_en.jpg (1800, 1736)
✓ mapa_umbra_en.jpg (1800, 1145)
✓ linea_tiempo_en.jpg (1800, 782)
✓ sitio_catia_la_mar_en.jpg (1800, 822)
✓ sitio_playa_grande_en.jpg (1800, 822)
✓ sitio_la_guaira_en.jpg (1800, 1048)
✓ sitio_caraballeda_en.jpg (1463, 1681)
Subir a GitHub docs/img / Upload to GitHub docs/img: /content/drive/My Drive/InSAR_LaGuaira_Resultados/para_github/docs/img

📝 Registro / log: /content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR/05_documentacion/logs/20261127_132905_00_Maestro_exportar_figuras_github.log


True

## Estado final / *Final status*

In [ ]:
maestro.estado(RAIZ)